# Task 3: Event Impact Modeling
Ethiopia Financial Inclusion Forecasting -- Selam Analytics

Builds the event x indicator association matrix and validates it against the observed Telebirr/M-Pesa mobile-money growth. Mirrors `src/impact_modeling.py`.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv('../data/processed/ethiopia_fi_enriched.csv')
events = df[df.record_type=='event'].copy()
links = df[df.record_type=='impact_link'].copy()
events['observation_date'] = pd.to_datetime(events['observation_date'])
links2 = links.drop(columns=['indicator_code'])
merged = links2.merge(events[['id','indicator']], left_on='parent_id', right_on='id')
merged = merged.rename(columns={'indicator_y':'event_name','related_indicator':'indicator_code'})
merged[['parent_id','event_name','indicator_code','impact_direction','impact_magnitude','lag_months','confidence']]

## Event -> Indicator association matrix

In [ ]:
matrix = merged.pivot_table(index='event_name', columns='indicator_code', values='impact_magnitude', aggfunc='sum', fill_value=0.0)
matrix

## Functional form: how effects build over time
We model each event's effect as a **logistic ramp**: zero before the event date, saturating to (approximately) its full documented `impact_magnitude` by `lag_months` after the event, rather than an instantaneous jump. Multiple simultaneous events combine additively per indicator.

In [ ]:
def event_effect(t_months, magnitude, lag_months):
    if t_months < 0:
        return 0.0
    k = 6.0 / max(lag_months, 1)
    x = t_months - lag_months / 2
    return magnitude / (1 + np.exp(-k * x / 6))

# sanity check: Telebirr (mag=4.2, lag=6) effect over time
for m in [0, 3, 6, 9, 12, 24]:
    print(m, 'months:', round(event_effect(m, 4.2, 6), 2))

## Validation against historical data
Telebirr launched May 2021; mobile money ownership rose from 4.7% (2021) to 9.45% (2024), an observed **+4.75pp** change. Summing the Telebirr + M-Pesa impact-link magnitudes (ramped) predicts **+8.7pp** -- an over-prediction. See `../reports/impact_model_validation.md` for full discussion and the resulting 0.55x discount factor applied to `ACC_MM_ACCOUNT` links in Task 4.

## Key assumptions & limitations
- Comparable-country evidence (Kenya M-Pesa, Bangladesh bKash, Nigeria NIBSS) is used where Ethiopian pre/post data is insufficient -- flagged as `confidence='medium'` or `'low'`.
- Effects are assumed additive across simultaneous events, which may overstate impact when events reinforce rather than independently add (addressed via the discount factor above).
- Lag windows are analyst judgment calibrated to the single validated case (Telebirr/mobile money); other indicator-event pairs are less well validated.